# 15.8 The Interactive Debugger — `pdb` and `breakpoint()`

**Prerequisites:** 15.7 Reading Failures, 15.3 pytest, 04 Functions  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- `breakpoint()` — the built-in way in, and `PYTHONBREAKPOINT` to switch it off
- The command set: stepping, inspecting, moving up and down the stack
- 🔴 **`n` is a command, not your variable** — and the two ways round it
- Conditional breakpoints, and breaking without editing the file
- **Post-mortem** debugging — inspecting a crash after it happened
- pytest integration: `--pdb`, `--trace`, `-l/--showlocals`
- 🔴 Why `--pdb` drops you in the *test* frame, not where the bug is
- When the debugger is the wrong tool

---

## Why a debugger beats `print`

`print` answers a question you have already formed. The debugger is for when you **do not yet
know what to ask** — it stops the program and lets you look at everything at once.

```
   print debugging                    debugger
   ───────────────                    ────────
   guess what matters                 stop, then look
   edit, re-run, read, repeat         one run, many questions
   1 variable per edit                every variable, every frame
   the loop is minutes                the loop is seconds
```

The turning point is the **third re-run**. If you have added `print` three times and still do
not understand, you are paying more than the debugger costs.

### Running the debugger from a notebook

`pdb` is interactive: it reads commands from stdin. A notebook cell cannot do that — which is
why every example below runs a real script in a **subprocess** and feeds it a scripted list of
commands. The output is genuine `pdb` output, not a transcript someone typed up.

> In your own work you will type these commands. In IPython/Jupyter there is also `%debug`,
> which opens a post-mortem session on the last exception — the single most useful notebook
> debugging command, covered at the end.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py158_"))


def write_script(name, source):
    path = WORK / name
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return path


def pdb_session(name, commands, args=(), env_extra=None, module_args=()):
    """Run a script under the debugger, feeding `commands` on stdin.

    `commands` is what you would type. The transcript below is real pdb output.
    """
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    if env_extra:
        env.update(env_extra)
    done = subprocess.run(
        [sys.executable, *module_args, str(WORK / name), *args],
        input="".join(c + "\n" for c in commands),
        cwd=WORK, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=120, env=env,
    )
    typed = " -> ".join(commands)
    return (f"typed: {typed}\n" + "-" * 70 + "\n"
            + (done.stdout + done.stderr).rstrip())


print("scratch:", WORK)

## `breakpoint()` — the way in

Since **Python 3.7** there is one built-in, no import needed:

```python
def retry_delay(attempt, base=1.0, ceiling=30.0):
    delay = base * 2 ** attempt
    breakpoint()                 # <- execution stops HERE, prompt appears
    return min(delay, ceiling)
```

Before 3.7 you wrote `import pdb; pdb.set_trace()`. You will still see that in older code; it
does the same thing.

🔴 **`breakpoint()` obeys the `PYTHONBREAKPOINT` environment variable**, which is the feature
nobody knows about:

| `PYTHONBREAKPOINT` | Effect |
|---|---|
| unset | `pdb.set_trace()` — the default |
| `0` | **every `breakpoint()` becomes a no-op** |
| `pudb.set_trace` | use a different debugger entirely |

That means a stray `breakpoint()` reaching production is survivable — set `PYTHONBREAKPOINT=0`
and the process runs straight through. It also means you can switch your whole team's debugger
without touching a line of code.

In [ ]:
write_script("backoff.py", r"""
    def retry_delay(attempt, base=1.0, ceiling=30.0):
        delay = base * 2 ** attempt
        breakpoint()
        return min(delay, ceiling)


    print("result:", retry_delay(3))
""")

print("### PYTHONBREAKPOINT=0 - the breakpoint is skipped entirely")
print(pdb_session("backoff.py", [], env_extra={"PYTHONBREAKPOINT": "0"}))

No prompt, no stop — the program simply ran. Now the same script with the
variable unset, driven by a list of commands.

In [ ]:
print(pdb_session("backoff.py", [
    "p delay",          # print an expression
    "p attempt",
    "args",             # every argument to the current function
    "pp locals()",      # pretty-print all locals
    "continue",         # let it run to the end
]))

## The command set

You need about a dozen. `pdb` accepts unambiguous abbreviations, shown in **bold**.

**Moving:**

| Command | Does |
|---|---|
| **`n`**ext | run the current line, stay in this function |
| **`s`**tep | run the current line, but **step into** any call |
| **`c`**ontinue | run until the next breakpoint or the end |
| **`r`**eturn | run until the current function returns |
| **`unt`**il *n* | run until a line **past** the current one — escapes loops |
| **`j`**ump *n* | 🔴 *change* which line runs next — skip or re-run code |

**Looking:**

| Command | Does |
|---|---|
| **`l`**ist | 11 lines around where you are; `ll` for the whole function |
| **`w`**here | the stack — same shape as a traceback (**15.7**) |
| **`a`**rgs | the current function's arguments |
| **`p`** *expr* | evaluate and print |
| **`pp`** *expr* | the same, pretty-printed — use for dicts and lists |
| **`display`** *expr* | show it automatically every time execution stops |

**Stack:**

| Command | Does |
|---|---|
| **`u`**p | move **out** one frame, towards the caller |
| **`d`**own | move back **in**, towards where it broke |

**Breakpoints:**

| Command | Does |
|---|---|
| **`b`** *file:line* | set a breakpoint; bare `b` lists them |
| **`b`** *line*, *condition* | break **only if** the condition is true |
| **`tbreak`** | break once, then remove itself |
| **`cl`**ear *n* | remove breakpoint *n* |

**Leaving:** `q`uit, or `c`ontinue.

The next cell steps through the function, moves up the stack, and lists source.

In [ ]:
write_script("schedule.py", r"""
    def retry_delay(attempt, base=1.0, ceiling=30.0):
        delay = base * 2 ** attempt
        return min(delay, ceiling)


    def schedule(attempts):
        breakpoint()
        first = retry_delay(0)
        return [first] + [retry_delay(n) for n in range(1, attempts)]


    print("schedule:", schedule(3))
""")

print(pdb_session("schedule.py", [
    "ll",               # the whole current function
    "where",            # how did we get here
    "n",                # 🔴 run the breakpoint() line itself; now we are ON the call
    "s",                # step INTO retry_delay
    "args",             # ...its three parameters
    "p base * 2 ** attempt",
    "u",                # back out to schedule
    "p attempts",
    "c",
]))

Read that transcript carefully — it contains the mistake everyone makes.

🔴 **`breakpoint()` stops you *on* its own line, which has not run yet.** The first `n` executes
the `breakpoint()` line and leaves you on `first = retry_delay(0)`. Only *then* does `s` step
**into** `retry_delay`, where `args` shows its three parameters.

Step too early and `s` behaves exactly like `n`, because the line you are standing on contains
no call. That is why a first attempt at this so often looks like "`s` doesn't work".

After that: `u` moved back out to `schedule`, where `attempts` is visible again — note it was
**not** visible from inside `retry_delay`, because each frame has its own locals (**04**). `c`
let it finish.

## 🔴 The trap: `n` is a command, not your variable

Every single-letter command shadows a variable of the same name. If your code has a variable
called `n`, `c`, `l`, `p`, `s`, `r`, `a`, `b`, `w`, `u`, `d`, `j` or `q` — typing it alone runs
the **command**.

Two ways out, both worth knowing:

| Form | Does |
|---|---|
| `p n` | print the *expression* `n` |
| `!n` | 🔴 execute the rest of the line as a **Python statement** — needed for assignment too |

`!` is the general escape: `!n = 10` reassigns a variable mid-session, which `n = 10` alone
cannot do.

In [ ]:
write_script("average.py", r"""
    def summarise(values):
        n = len(values)
        total = sum(values)
        breakpoint()
        return total / n


    print("mean:", summarise([3, 5, 8]))
""")

print(pdb_session("average.py", [
    "n",            # 🔴 this STEPS - it does not print `n`
    "p n",          # this prints it
    "!n",           # so does this
    "p total / n",
    "!n = 4",       # ! also lets you ASSIGN - watch the answer change
    "p n",
    "c",
]))

Look at the last line of that output. The mean of `[3, 5, 8]` is
`5.333...`, but the program printed **`4.0`** — because `!n = 4` changed the divisor while it
was stopped.

That is `pdb`'s most powerful and most dangerous feature: you are not watching a recording, you
are **editing a running program**. Use it to test a hypothesis in seconds ("if this were 4,
would the bug go away?") — and never forget you have done it.

## Breaking without editing the file

Editing a file to insert `breakpoint()` is fine for your own code. For a dependency — or a file
you must not touch — set the breakpoint from the command line instead:

```bash
python -m pdb -c "break retry.py:12" -c "continue" myscript.py
```

`-c` runs a command before the program starts, and you can pass several. A **conditional**
breakpoint is how you catch the one iteration that matters out of a thousand.

In [ ]:
write_script("batch.py", r"""
    def apply_cap(delay, cap=30.0):
        return min(delay, cap)


    def run_batch(jobs):
        results = []
        for job_id, attempt in jobs:
            delay = apply_cap(1.0 * 2 ** attempt)
            results.append((job_id, delay))
        return results


    print(run_batch([("build-1", 1), ("build-2", 3), ("build-3", 9)]))
""")

# Break ONLY on the iteration where the ceiling actually bites.
print(pdb_session("batch.py", [
    "break batch.py:8, attempt > 4",     # file:line, condition
    "continue",
    "p job_id, attempt",
    "p 1.0 * 2 ** attempt",
    "p apply_cap(1.0 * 2 ** attempt)",
    "quit",             # quit here: letting it finish makes `-m pdb` RESTART the program
], module_args=("-m", "pdb")))

The debugger ran straight past `build-1` and `build-2` and stopped only on
`build-3`, where `attempt > 4` — then showed that `512.0` was capped to `30.0`.

🔴 A conditional breakpoint is what makes the debugger usable on real data. Without it, you
press `c` two hundred times and give up.

> **Why `quit` and not `continue` here.** Under `python -m pdb`, when the program finishes,
> `pdb` prints *"The program finished and will be restarted"* and starts it again from the top,
> so you can debug another run. Handy in a real session, confusing in a transcript.

## Post-mortem: debugging a crash that already happened

You do not have to predict where a bug is. When something raises, you can open a debugger
**on the corpse** — the stack is still intact, with every local variable.

| How | When |
|---|---|
| `python -m pdb -c continue script.py` | run it; on a crash, drop into post-mortem |
| `pdb.post_mortem()` | in an `except` block |
| `pdb.pm()` | after the fact, on the last exception |
| `%debug` | in IPython/Jupyter — on the last exception in the session |

In [ ]:
write_script("parse.py", r"""
    def parse_budgets(raw):
        return {key: int(value)
                for key, value in (pair.split("=") for pair in raw.split(","))}


    print(parse_budgets("cpu=4,mem=8,disk"))
""")

# -c continue runs the program; when it crashes, pdb takes over the dead frame.
print(pdb_session("parse.py", [
    "p raw",                 # the input that broke it
    "p raw.split(',')",      # ...and why
    "where",
    "quit",
], module_args=("-m", "pdb", "-c", "continue")))

No `breakpoint()` was added and the program was not modified — yet you can
see `raw` and evaluate expressions against it, *after* the crash. `raw.split(",")` shows the
third element is `'disk'` with no `=`, so `pair.split("=")` returned one item and the unpacking
failed.

> **Version note.** Post-mortem uses `sys.last_exc` (3.12+, previously `sys.last_value`). In a
> plain REPL, `import pdb; pdb.pm()` opens the same session on the last traceback.

## pytest and the debugger

This is where **15.3** and this notebook meet. Three flags:

| Flag | Does |
|---|---|
| `--pdb` | on failure, drop into post-mortem **at the failing test** |
| `--trace` | break at the **start** of each selected test |
| `-l` / `--showlocals` | no debugger — just print every local in the failing frame |

`--showlocals` is the one to try first: it is non-interactive, works in CI, and answers most
questions on its own.

In [ ]:
(WORK / "pt").mkdir(exist_ok=True)
(WORK / "pt" / "test_backoff.py").write_text(textwrap.dedent(r"""
    def apply_cap(delay, cap):
        return min(delay, cap)


    def test_ceiling_is_applied():
        attempt = 5
        base = 1.0
        delay = base * 2 ** attempt
        assert apply_cap(delay, 30.0) == 32.0
""").lstrip(), encoding="utf-8")


def pytest_in(*args, stdin=""):
    done = subprocess.run(
        [sys.executable, "-m", "pytest", "--no-header", "-p", "no:cacheprovider", *args],
        cwd=WORK / "pt", input=stdin, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=180,
    )
    return (f"$ pytest {' '.join(args)}\n" + "-" * 70 + "\n"
            + (done.stdout + done.stderr).rstrip())


print(pytest_in("-q", "--showlocals", "--tb=long"))

`--showlocals` printed the locals table — `attempt = 5`, `base = 1.0`,
`delay = 32.0` — directly under the failing assertion. No interaction, works in CI logs.

Now the interactive version.

In [ ]:
print(pytest_in("-q", "--pdb",
                stdin="p delay\np cap\np apply_cap(delay, 30.0)\nq\n"))

🔴 **Read what happened to `p cap`.** It raised
`*** NameError: name 'cap' is not defined` — even though `cap` is a parameter of `apply_cap`,
the function the test is complaining about.

The reason is worth internalising, because it applies to **every** post-mortem session:

> **Post-mortem gives you the frames on the traceback — not every frame that ran.**

`apply_cap` **returned successfully**. It is not on the traceback; the `assert` in the *test*
is what raised. So `apply_cap`'s frame no longer exists and nothing can bring it back — `u` and
`d` walk the frames that *are* there, which above the test are all pytest internals.

`delay` works because it is a local of the test frame. And the practical move when you need to
know what a returned function did is the third command above: **call it again from the
prompt** — `p apply_cap(delay, 30.0)` returns `30.0`, which is the bug, established without
re-running anything.

If you genuinely need to stop *inside* `apply_cap`, you must break before it returns:
`--trace`, or a `breakpoint()` in the function itself.

## 🔴 When *not* to use the debugger

The debugger is not always the right instrument, and reaching for it reflexively wastes time.

| Situation | Better tool |
|---|---|
| The bug is intermittent | **logging** (**15.7**) — you cannot sit at a prompt waiting for it |
| It only happens in production | logging, and a metric |
| It only happens under load or concurrency | 🔴 the debugger **changes the timing** and the bug vanishes (**15.9**) |
| You already know the value you want | `print`, or a test |
| You need the answer to stay answered | **a test** (**15.1**) |
| The process is *hung*, not crashed | `faulthandler` (**15.7**) |

And the honest limit: **stopping the world perturbs it**. Any bug involving timing, threads
(**12.2**), sockets with timeouts (**11.2**) or real users can disappear the moment you attach
a debugger. That class of bug is what **15.9** is about.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Typing a bare variable name that is also a command.** `n`, `c`, `l`, `p`, `s`, `r`, `a`, `b`, `w`, `u`, `d`, `j`, `q` all run commands. Use `p name`, or `!name`.
2. 🔴 **Expecting `pytest --pdb` to land where the bug is.** It lands in the *test* frame; use `d` to move down into the failing call.
3. **Committing a stray `breakpoint()`.** It will hang CI, silently, until the job times out. Set `PYTHONBREAKPOINT=0` in CI as a safety net — and grep before you push.
4. **Pressing `c` repeatedly to reach the interesting iteration.** Use a conditional breakpoint: `b file.py:12, attempt > 4`.
5. **Confusing `n` and `s`.** `n` runs a call as one step; `s` goes inside it. Stepping into every library call is how a session becomes unusable.
6. **Forgetting you used `!x = ...` to change a value.** The program's behaviour is now yours, not its.
7. **Using the debugger on a concurrency or timing bug.** Stopping the world hides it (**15.9**).
8. **Debugging instead of writing a failing test.** The debugger tells you the answer once; a test keeps telling you.

## Best Practices

- Reach for the debugger on the **third** `print` — before that, `print` is genuinely faster.
- Try `--showlocals` before `--pdb`: non-interactive, CI-safe, and usually enough.
- Use conditional breakpoints rather than pressing `c` until something interesting happens.
- Learn `u` / `d` early — most confusion in a debugging session is being in the wrong frame.
- Use `python -m pdb -c continue` for post-mortem on a crash you cannot easily reproduce.
- Use `display expr` to watch a value change as you step, rather than re-typing `p expr`.
- Set `PYTHONBREAKPOINT=0` in CI so a stray `breakpoint()` cannot hang the build.
- When the debugger tells you the answer, **write the test** that would have caught it (**15.1**).

## Practice Exercises

Try these before moving on.

1. Add `breakpoint()` to a function of your own, then use `args`, `pp locals()`, `w` and `u` to answer: what called this, and with what?
2. 🔴 Write a function with a local variable named `l`. Stop in it and try to print `l` three ways. Which work?
3. Use `!` to change a variable mid-session so a failing function returns the right answer. What does that tell you about where the bug is?
4. Take the `parse_budgets` crash from this notebook and fix it *without* re-running the program — work out the fix entirely from a post-mortem session.
5. Set a conditional breakpoint that fires only on the 500th iteration of a loop. Compare with how long pressing `c` would take.
6. Run any failing test from **15.3** with `--showlocals`, then `--pdb`, then `--trace`. When would you use each?
7. 🔴 In `pytest --pdb`, reproduce the `NameError` on a variable that lives in the called function, then reach it with `d`. Write down what `w` showed you.
8. **Interview question:** how would you debug a bug that only appears in production, once a day, on one of twenty servers? (The answer is not a debugger — see **15.9**.)

---

## Version notes

| Version | Change |
|---|---|
| **3.14** | `pdb` gained a `--pid` option to attach to a **running process** (`python -m pdb -p PID`) — previously this needed a third-party tool |
| **3.13** | `pdb` supports multi-line statements at the prompt; better `breakpoint()` behaviour in threads |
| **3.12** | `sys.last_exc` added; `pdb.pm()` uses it |
| **3.7** | 🔴 **`breakpoint()` built in**, and `PYTHONBREAKPOINT`. Before this: `import pdb; pdb.set_trace()` |
| **3.2** | `python -m pdb -c command` — the flag used for the post-mortem examples here |

> **Beyond `pdb`.** `ipdb` (better completion and colour), `pudb` (a full-screen terminal UI),
> and the debuggers built into VS Code and PyCharm all speak the same concepts — breakpoints,
> stepping, frames, watches. Learn `pdb` and the rest are a keyboard shortcut away. All of them
> honour `PYTHONBREAKPOINT`.

## Where next

| Notebook | Covers |
|---|---|
| **15.9 Debugging in Practice** | strategy, bisection, and bugs the debugger cannot catch |

## Related

- **15.7** — tracebacks, logging and `faulthandler`; the non-interactive half
- **15.3 pytest** — `--pdb`, `--trace`, `--showlocals`, and node IDs for running one test
- **15.1** — writing the test the debugger just taught you to write
- **12.2 Threading** — bugs that the debugger's own timing hides
- **04 Functions** — frames, scope and closures, which is what `u`/`d` walk through